# Structured output

Đầu ra có cấu trúc cho phép các agent trả về dữ liệu theo một định dạng cụ thể, dễ dự đoán. Thay vì phải parse các câu trả lời bằng ngôn ngữ tự nhiên, bạn sẽ nhận được dữ liệu có cấu trúc dưới dạng các JSON object, [Pydantic model](https://docs.pydantic.dev/latest/concepts/models/#basic-model-usage), hoặc các dataclass mà ứng dụng của bạn có thể sử dụng trực tiếp.

<div class="alert alert-success">

Trang này hướng dẫn cách sử dụng đầu ra có cấu trúc với các agent thông qua `create_agent`. Để sử dụng đầu ra có cấu trúc trực tiếp trên một model (không thông qua agent), hãy xem [Models - Structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output).

</div>

Hàm [`create_agent`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) của LangChain xử lý đầu ra có cấu trúc một cách tự động. Người dùng thiết lập schema mong muốn, và khi model tạo ra dữ liệu có cấu trúc, dữ liệu đó sẽ được nắm bắt, xác thực, và trả về trong key `'structured_response'` của state agent.

```python
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
        None,
    ]
)
```

## Định dạng phản hồi

Sử dụng `response_format` để kiểm soát cách agent trả về dữ liệu có cấu trúc:

* **`ToolStrategy[StructuredResponseT]`**: Sử dụng tool calling để tạo đầu ra có cấu trúc.
* **`ProviderStrategy[StructuredResponseT]`**: Sử dụng tính năng đầu ra có cấu trúc gốc của provider.
* **`type[StructuredResponseT]`**: Loại schema - tự động chọn chiến lược tốt nhất dựa trên khả năng của model.
* **`None`**: Không yêu cầu đầu ra có cấu trúc một cách rõ ràng.

Khi một loại schema được cung cấp trực tiếp, LangChain sẽ tự động chọn:

* `ProviderStrategy` nếu model và provider được chọn có hỗ trợ đầu ra có cấu trúc gốc (ví dụ: [OpenAI](https://docs.langchain.com/oss/python/integrations/providers/openai), [Anthropic (Claude)](https://docs.langchain.com/oss/python/integrations/providers/anthropic), hoặc [xAI (Grok)](https://docs.langchain.com/oss/python/integrations/providers/xai)).
* `ToolStrategy` cho tất cả các model khác.

<div class="alert alert-warning">

Các JSON schema dict phải được bọc trong một chiến lược rõ ràng (`ProviderStrategy` hoặc `ToolStrategy`). Chúng sẽ không được tự động nhận diện nếu truyền trực tiếp vào `response_format`.

</div>

<div class="alert alert-info">

Việc hỗ trợ các tính năng đầu ra có cấu trúc gốc sẽ được đọc động từ [profile data](https://docs.langchain.com/oss/python/langchain/models#model-profiles) của model nếu bạn dùng `langchain>=1.1`. Nếu không có sẵn dữ liệu, hãy sử dụng một điều kiện khác hoặc chỉ định thủ công:

```python
custom_profile = {
    "structured_output": True,
    # ...
}
model = init_chat_model("...", profile=custom_profile)
```

Nếu có chỉ định tool, model phải hỗ trợ việc sử dụng đồng thời cả tool và đầu ra có cấu trúc.

</div>

Phản hồi có cấu trúc sẽ được trả về trong key `structured_response` thuộc state cuối cùng của agent.

## Chiến lược từ provider

Một số model provider hỗ trợ trực tiếp đầu ra có cấu trúc thông qua API của họ (ví dụ: OpenAI, xAI (Grok), Gemini, Anthropic (Claude)). Đây là phương pháp đáng tin cậy nhất khi được hỗ trợ.

Để sử dụng chiến lược này, hãy cấu hình `ProviderStrategy`:

```python
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    strict: bool | None = None
```

<div class="alert alert-info">

Tham số `strict` yêu cầu phiên bản `langchain>=1.2`.

</div>

`schema` (required): Schema định nghĩa định dạng của đầu ra có cấu trúc. Hỗ trợ:

* **Pydantic model**: Các class kế thừa từ `BaseModel` kèm theo xác thực trường dữ liệu. Trả về một instance Pydantic đã được xác thực.
* **Dataclass**: Các dataclass của Python có chú thích kiểu dữ liệu. Trả về dict.
* **TypedDict**: Các class dictionary có định kiểu. Trả về dict.
* **JSON Schema**: Dictionary chứa đặc tả JSON schema. Phải bao gồm các key `title` và `description` ở cấp cao nhất. Trả về dict.

`strict`: Tham số boolean tùy chọn để bật tính năng tuân thủ schema nghiêm ngặt. Được hỗ trợ bởi một số provider (ví dụ: [OpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) và [xAI](https://docs.langchain.com/oss/python/integrations/chat/xai)). Mặc định là `None` (vô hiệu hóa).

LangChain sẽ tự động sử dụng `ProviderStrategy` khi bạn truyền trực tiếp một loại schema vào [`create_agent.response_format`](https://reference.langchain.com/python/langchain/agents/factory/create_agent) và model có hỗ trợ đầu ra có cấu trúc gốc:

In [6]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Thông tin liên hệ của một người."""
    name: str = Field(description="Tên của người đó")
    email: str = Field(description="Địa chỉ email của người đó")
    phone: str = Field(description="Số điện thoại của người đó")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo  # Tự động chọn ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Trích xuất thông tin liên hệ từ: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

Đầu ra có cấu trúc gốc từ provider mang lại độ tin cậy cao và khả năng xác thực nghiêm ngặt vì chính model provider sẽ ép tuân thủ schema. Hãy sử dụng tính năng này khi có sẵn.

<div class="alert alert-info">

Nếu provider có hỗ trợ sẵn đầu ra có cấu trúc cho model bạn chọn, việc viết `response_format=ProductReview` sẽ có chức năng tương đương với việc viết `response_format=ProviderStrategy(ProductReview)`.

Trong cả hai trường hợp, nếu đầu ra có cấu trúc không được hỗ trợ, agent sẽ tự động chuyển về chiến lược tool calling.

</div>

## Chiến lược gọi tool

Đối với các model không hỗ trợ đầu ra có cấu trúc gốc, LangChain sử dụng tool calling để đạt được kết quả tương tự. Phương pháp này hoạt động với tất cả các model có hỗ trợ tool calling (hầu hết các model hiện đại).

Để sử dụng chiến lược này, hãy cấu hình `ToolStrategy`:

```python
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```

`schema` (required): Schema định nghĩa định dạng của đầu ra có cấu trúc. Hỗ trợ:

* **Pydantic model**: Các class kế thừa từ `BaseModel` kèm theo xác thực trường dữ liệu. Trả về một instance Pydantic đã được xác thực.
* **Dataclass**: Các dataclass của Python có chú thích kiểu dữ liệu. Trả về dict.
* **TypedDict**: Các class dictionary có định kiểu. Trả về dict.
* **JSON Schema**: Dictionary chứa đặc tả JSON schema. Phải bao gồm các key `title` và `description` ở cấp cao nhất. Trả về dict.
* **Union type**: Nhiều tùy chọn schema. Model sẽ chọn schema phù hợp nhất dựa trên ngữ cảnh.

`tool_message_content`: Nội dung tùy chỉnh cho tool message được trả về khi đầu ra có cấu trúc được tạo. Nếu không được cung cấp, mặc định sẽ là một thông báo hiển thị dữ liệu phản hồi có cấu trúc.

`handle_errors`: Chiến lược xử lý lỗi khi xác thực đầu ra có cấu trúc thất bại. Mặc định là `True`.

* **`True`**: Bắt tất cả các lỗi với template lỗi mặc định
* **`str`**: Bắt tất cả các lỗi và hiển thị thông báo tùy chỉnh này
* **`type[Exception]`**: Chỉ bắt loại ngoại lệ này và hiển thị thông báo mặc định
* **`tuple[type[Exception], ...]`**: Chỉ bắt các loại ngoại lệ này và hiển thị thông báo mặc định
* **`Callable[[Exception], str]`**: Hàm tùy chỉnh dùng để trả về thông báo lỗi
* **`False`**: Không thử lại, để các ngoại lệ tự động ném ra

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Phân tích một bài đánh giá sản phẩm."""
    rating: int | None = Field(description="Điểm đánh giá của sản phẩm", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="Cảm xúc của bài đánh giá")
    key_points: list[str] = Field(description="Các điểm chính của bài đánh giá. Viết thường, mỗi điểm từ 1-3 từ.")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Phân tích bài đánh giá này: 'Sản phẩm tuyệt vời: 5 trên 5 sao. Giao hàng nhanh, nhưng đắt'"}]
})
result["structured_response"]

ProductReview(rating=5, sentiment='positive', key_points=['tuyệt vời', 'giao hàng nhanh', 'giá đắt'])

### Tùy chỉnh nội dung tool message

Tham số `tool_message_content` cho phép bạn tùy chỉnh thông báo xuất hiện trong lịch sử hội thoại khi đầu ra có cấu trúc được tạo ra:

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Các mục hành động được trích xuất từ biên bản cuộc họp."""
    task: str = Field(description="Nhiệm vụ cụ thể cần hoàn thành")
    assignee: str = Field(description="Người chịu trách nhiệm cho nhiệm vụ")
    priority: Literal["low", "medium", "high"] = Field(description="Mức độ ưu tiên")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Đã ghi nhận mục hành động và thêm vào ghi chú cuộc họp!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "Từ cuộc họp của chúng ta: Sarah cần cập nhật tiến độ dự án càng sớm càng tốt"}]
})

{'messages': [HumanMessage(content='Từ cuộc họp của chúng ta: Sarah cần cập nhật tiến độ dự án càng sớm càng tốt', additional_kwargs={}, response_metadata={}, id='80fa2a53-73a4-4498-9ecd-2b6a7dc70148'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'MeetingAction', 'arguments': '{"priority": "high", "assignee": "Sarah", "task": "C\\u1eadp nh\\u1eadt ti\\u1ebfn \\u0111\\u1ed9 d\\u1ef1 \\u00e1n"}'}, '__gemini_function_call_thought_signatures__': {'call_984600': 'Eo8KCowKARFNMg8PcIfQtuDVLCi0SvUU04SUMl/XqujL8muGxwqyBPiOCr6tijhhhFOTNYpBWL7qAWLxImRa+JsnHlS/ar+/sqKv6EPSArFaMS4H9XuDPAVct9DzAlQSuknRvaY1teAwpI/TB/JGLKLuSdGxVkF6Lb3jFPKn3LZH4VzUgUhzv5vS9aAlIUhGA+I7MJ6fr7dBYfD8S68iaM0oXff6pe/kImZrnL8Vagae9i/Q3Fb9IZ5AjNmZ7iMux4kV4JQGjsFc62DwjThjJWnVJpfAmPMlBSOu8LZ1XYDXpsjDoe4hBdC+oXTCN2ufbYufqyqnta1YzGGwHU9V+RDjLrmcMgcpwrSSW17UZeikJG/7UQ50QpOF9sA9YB6HNOpmDtTFRDel+UrmdIh63eWtdpTjIGTkmiMdfdApenNR48nVa580e812ga0NNYHoNcYLRrhXd+s+JMIokFDt5hZeFburFU0d+JdYiyQ39mYN4fueDc69iLxHNepa0MMb

Nếu không có `tool_message_content`, [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) cuối cùng của chúng ta sẽ là:

```
================================= Tool Message =================================
Name: MeetingAction

Returning structured response: {'task': 'Cập nhật tiến độ dự án', 'assignee': 'Sarah', 'priority': 'high'}
```

### Xử lý lỗi

Các model có thể mắc sai lầm khi tạo đầu ra có cấu trúc qua tool calling. LangChain cung cấp các cơ chế retry thông minh để xử lý những lỗi này một cách tự động.

#### Lỗi trả về nhiều đầu ra có cấu trúc

Khi model gọi sai nhiều công cụ đầu ra có cấu trúc cùng lúc, agent sẽ cung cấp phản hồi báo lỗi trong một [`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage) và yêu cầu model thử lại:

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Tên của người đó")
    email: str = Field(description="Địa chỉ email")

class EventDetails(BaseModel):
    event_name: str = Field(description="Tên của sự kiện")
    date: str = Field(description="Ngày diễn ra sự kiện")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Mặc định: handle_errors=True
)

agent.invoke({
    "messages": [{"role": "user", "content": "Trích xuất thông tin: John Doe (john@email.com) đang tổ chức Tech Conference vào ngày 15 tháng 3"}]
})

{'messages': [HumanMessage(content='Trích xuất thông tin: John Doe (john@email.com) đang tổ chức Tech Conference vào ngày 15 tháng 3', additional_kwargs={}, response_metadata={}, id='33f6de2b-6edb-4e1d-8fd5-acfa0fb439ae'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'EventDetails', 'arguments': '{"event_name": "Tech Conference", "date": "15 th\\u00e1ng 3"}'}, '__gemini_function_call_thought_signatures__': {'call_973330': 'EswJCskJARFNMg8mxM7WUcN4V0lOhOhcfOihE9Uw1ep7lZqKXNvF5sHS9ZIL0MgpsMsbR227UOYOv8fJEvzFPHzhjFUPYCLJdFBA1Kt9kH2Z5JWvgKJVnklccg1jFNMxsWPCL5TDdDeGwXBsVF6Y+425x0ZYfFRawFcHloeQoMA+B0wVD9HV49oMRR2onVjgtIqzeOi7DqXA80oSjGg55EuyMWVa3CkKVMko97U9VivL24YTVOaoGQ3pFyxISckltGtgxq8pcZQU/+mK6ujKJbGjzJEXgdd4N9oxJ3T8qoqSkualmQL6Ao22Jlu4OdHKZo0wEJA0abtv6MQN1mLd/FeBB+kq/a7bcCrmv6WL9x7SVedrhpxsS12jKEkO/wXKFggSX7nz/qOAHIGw1On7putG6/+mvHrwZIvGtvtfVX7yvfu58u8WdzMHl3xXFSQfPMDTTY8lJs7L1mjXtLCbMOJRmPuM0umDkEqqQ7Vs1XH5c0Uy90DOvSX6erHb9YYS1NZ23HFrjCci8tgC47vtuaI/i4OJIX1K0+jCI

#### Lỗi xác thực schema

Khi đầu ra có cấu trúc không khớp với schema mong đợi, agent sẽ cung cấp phản hồi lỗi chi tiết:

In [15]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Điểm đánh giá", ge=1, le=5)
    comment: str = Field(description="Bình luận đánh giá")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Mặc định: handle_errors=True
    system_prompt="Bạn là một trợ lý hữu ích chuyên phân tích các bài đánh giá sản phẩm. Không được tự bịa ra bất kỳ trường dữ liệu hay giá trị nào."
)

agent.invoke({
    "messages": [{"role": "user", "content": "Phân tích nội dung này: Sản phẩm tuyệt vời, 10/10!"}]
})

{'messages': [HumanMessage(content='Phân tích nội dung này: Sản phẩm tuyệt vời, 10/10!', additional_kwargs={}, response_metadata={}, id='4d1aa37a-8837-4dd3-90f4-4f9859cbda55'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ProductRating', 'arguments': '{"comment": "S\\u1ea3n ph\\u1ea9m tuy\\u1ec7t v\\u1eddi, 10/10!", "rating": 10}'}, '__gemini_function_call_thought_signatures__': {'call_2340611': 'El4KXAERTTIPWwrl4vKodxgVgLjidurbhgOFz3L/92Ynjto/n3dcyohHvrp3wbN9OpYKdzIDZN52d6SQvfLHeDVMMBdSiaczpQYVqMfaJO1P1PODaRDuPHfmOWCqY9jn'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a06b3f-74da-7e90-a982-215b68604005-0', tool_calls=[{'name': 'ProductRating', 'args': {'comment': 'Sản phẩm tuyệt vời, 10/10!', 'rating': 10}, 'id': 'call_2340611', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 32, 'total_tokens': 151,

#### Các chiến lược xử lý lỗi

Bạn có thể tùy chỉnh cách xử lý các lỗi bằng cách sử dụng tham số `handle_errors`:

**Thông báo lỗi tùy chỉnh:**

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors="Vui lòng cung cấp điểm đánh giá hợp lệ từ 1-5 và kèm theo bình luận."
)

Nếu `handle_errors` là một string, agent sẽ *luôn luôn* yêu cầu model thử lại bằng một thông báo công cụ cố định:

```
================================= Tool Message =================================
Name: ProductRating

Vui lòng cung cấp điểm đánh giá hợp lệ từ 1-5 và kèm theo bình luận.
```

**Chỉ xử lý các ngoại lệ cụ thể:**

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Chỉ thử lại khi gặp ValueError, ném ra các lỗi khác
)

Nếu `handle_errors` là một loại ngoại lệ, agent sẽ chỉ retry (sử dụng thông báo lỗi mặc định) nếu ngoại lệ được ném ra đúng với loại đã chỉ định. Trong tất cả các trường hợp khác, ngoại lệ đó sẽ được ném ra ngoài.

**Xử lý nhiều loại ngoại lệ:**

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Thử lại khi gặp ValueError và TypeError
)

Nếu `handle_errors` là một tuple chứa các ngoại lệ, agent sẽ chỉ retry (sử dụng thông báo lỗi mặc định) nếu ngoại lệ được ném ra thuộc một trong những loại đã chỉ định. Trong tất cả các trường hợp khác, ngoại lệ sẽ được ném ra ngoài.

**Hàm xử lý lỗi tùy chỉnh:**

In [ ]:
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "Có vấn đề với định dạng. Vui lòng thử lại."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Nhiều đầu ra có cấu trúc được trả về. Hãy chọn cái phù hợp nhất."
    else:
        return f"Lỗi: {str(error)}"


agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    response_format=ToolStrategy(
                        schema=Union[ContactInfo, EventDetails],
                        handle_errors=custom_error_handler
                    )  # Mặc định: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Trích xuất thông tin: John Doe (john@email.com) đang tổ chức Tech Conference vào ngày 15 tháng 3"}]
})

for msg in result['messages']:
    # Nếu message thực sự là một đối tượng ToolMessage (không phải dict), hãy kiểm tra tên class của nó
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # Nếu message là một dictionary hoặc bạn muốn có phương án dự phòng
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])

Khi gặp `StructuredOutputValidationError`:

```
================================= Tool Message =================================
Name: ToolStrategy

Có vấn đề với định dạng. Vui lòng thử lại.
```

Khi gặp `MultipleStructuredOutputsError`:

```
================================= Tool Message =================================
Name: ToolStrategy

Nhiều đầu ra có cấu trúc được trả về. Hãy chọn cái phù hợp nhất.
```

Đối với các lỗi khác:

```
================================= Tool Message =================================
Name: ToolStrategy

Lỗi: <thông báo lỗi>
```

**Không xử lý lỗi:**

In [ ]:
response_format = ToolStrategy(
    schema=ProductRating,
    handle_errors=False  # Tất cả các lỗi sẽ được ném ra
)